# Fast Sign Recognition: A Small Proof of Concept

We test three simple CPU methods on ISL and Nepali clips separately.

| Method | What it does |
| --- | --- |
| All frames | Describe every frame, average into eight time bins, then classify. |
| Uniform 8 | Describe eight spaced frames, then classify. |
| Adaptive 4 to 8: our candidate | Start with four of those frames. If the score gap is small, describe the remaining frames and use the eight-frame classifier. Reuse work already done. |

All three use small gradient features and linear classifiers trained with SGD hinge loss. These are not deep models. The four-frame features are expanded to eight positions so feature size stays the same. Frame-difference features describe change.

Our candidate adapts known conditional-compute ideas to this small sign task. Early exits and frame sampling are not new. Novelty of this exact combination is **not verified**. This is word/alphabet classification, not sentence translation.

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
import speed_poc

ROOT = Path.cwd()
DATA = speed_poc.datasets(ROOT)
speed_poc.smoke_test(ROOT)
RERUN = False
REPORTS = speed_poc.run(ROOT, repeats=2) if RERUN else {
    name: json.loads((ROOT / 'results' / 'speed_poc' / name.lower() / 'metrics.json').read_text())
    for name in DATA
}
print('Python:', sys.executable)
print('Hardware:', REPORTS['ISL']['device'])
print('OpenCV:', REPORTS['ISL']['opencv'], '| pandas:', pd.__version__)
print('RERUN=False shows the measured full run. Set True to retrain and remeasure; allow several minutes.')
for language, records in DATA.items():
    counts = pd.Series([record['label'] for record in records]).value_counts().sort_index()
    print(f'{language}: {len(records)} clips, {len(counts)} classes, {len(set(record["group"] for record in records))} groups')
    display(counts.rename('clips per class').to_frame())

In [ ]:
import pickle
import cv2
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

VIS_LANGUAGE = 'ISL'
VIS_CLIP = 0
visual_test_records = [record for record in DATA[VIS_LANGUAGE]
                       if VIS_LANGUAGE == 'ISL' or record['split'] == 'test']
visual_record = visual_test_records[VIS_CLIP]
visual_fold = next(fold for fold in REPORTS[VIS_LANGUAGE]['folds']
                   if visual_record['group'] in fold['test_groups'])
visual_model_path = ROOT / 'results' / 'speed_poc' / VIS_LANGUAGE.lower() / f"fold_{visual_fold['fold']}.pkl"
with visual_model_path.open('rb') as visual_stream:
    visual_saved = pickle.load(visual_stream)

visual_gray = speed_poc.decode(visual_record['path'])
visual_small, visual_full = speed_poc.frame_indices(len(visual_gray))
visual_features = speed_poc.ClipFeatures(visual_gray)
visual_coarse_prediction, visual_gap = speed_poc.decision(
    visual_saved['models']['coarse'], visual_features.feature(visual_small))
visual_rescued = bool(visual_gap[0] < visual_saved['threshold'])
visual_adaptive_indices = visual_full if visual_rescued else visual_small
visual_adaptive_prediction = (speed_poc.decision(
    visual_saved['models']['full'], visual_features.feature(visual_full))[0]
    if visual_rescued else visual_coarse_prediction)
visual_dense_prediction = speed_poc.decision(
    visual_saved['models']['dense'], visual_features.feature(np.arange(len(visual_gray))))[0]
visual_uniform_prediction = speed_poc.decision(
    visual_saved['models']['full'], visual_features.feature(visual_full))[0]

visual_rgb_frames = []
visual_capture = cv2.VideoCapture(str(visual_record['path']))
try:
    while True:
        visual_ok, visual_bgr = visual_capture.read()
        if not visual_ok:
            break
        visual_rgb = cv2.cvtColor(visual_bgr, cv2.COLOR_BGR2RGB)
        visual_height, visual_width = visual_rgb.shape[:2]
        visual_rgb_frames.append(cv2.resize(visual_rgb,
            (240, max(1, round(visual_height * 240 / visual_width)))))
finally:
    visual_capture.release()
assert len(visual_rgb_frames) == len(visual_gray)
visual_sets = [set(range(len(visual_gray))), set(visual_full), set(visual_adaptive_indices)]
visual_predictions = [visual_dense_prediction[0], visual_uniform_prediction[0], visual_adaptive_prediction[0]]
visual_colors = ['#238443', '#2171b5', '#d94801']
visual_fig, visual_axes = plt.subplots(2, 3, figsize=(12, 5.5),
                                     gridspec_kw={'height_ratios': [5, 1]}, constrained_layout=True)
visual_images, visual_labels, visual_cursors = [], [], []
for visual_column, (visual_name, visual_selected, visual_color) in enumerate(
        zip(speed_poc.METHODS, visual_sets, visual_colors)):
    visual_axis = visual_axes[0, visual_column]
    visual_images.append(visual_axis.imshow(visual_rgb_frames[0]))
    visual_axis.set_title(f'{visual_name}: {len(visual_selected)} frame descriptors\n'
                          f'Prediction: {visual_predictions[visual_column]}', fontsize=11)
    visual_labels.append(visual_axis.text(0.5, 0.97, '', transform=visual_axis.transAxes,
                                         ha='center', va='top', color='white', fontsize=10,
                                         bbox={'facecolor': visual_color, 'alpha': 0.9, 'pad': 5}))
    visual_axis.axis('off')
    visual_timeline = visual_axes[1, visual_column]
    visual_timeline.scatter(range(len(visual_gray)), np.zeros(len(visual_gray)), s=9, color='#cccccc')
    visual_timeline.scatter(sorted(visual_selected), np.zeros(len(visual_selected)), s=30, color=visual_color)
    visual_cursors.append(visual_timeline.axvline(0, color='black', linewidth=1))
    visual_timeline.set(xlim=(-1, len(visual_gray)), ylim=(-0.4, 0.4), xlabel='Frame index (zero-based)', yticks=[])
    visual_timeline.spines[['left', 'right', 'top']].set_visible(False)
visual_fig.suptitle(f"{VIS_LANGUAGE}: {visual_record['path'].name} | Actual label: {visual_record['label']}\n"
                    f"Adaptive rescue: {visual_rescued} | Score gap: {visual_gap[0]:.2f} | Gate: {visual_saved['threshold']:.2f}",
                    fontsize=11)

def update_method_preview(frame_index):
    for column_index, selected in enumerate(visual_sets):
        used = frame_index in selected
        visual_images[column_index].set_data(visual_rgb_frames[frame_index])
        visual_images[column_index].set_alpha(1.0 if used else 0.35)
        visual_labels[column_index].set_text(
            f'Frame {frame_index}: ' + ('FEATURE USED' if used else 'FEATURE SKIPPED'))
        visual_labels[column_index].get_bbox_patch().set_facecolor(
            visual_colors[column_index] if used else '#666666')
        visual_cursors[column_index].set_xdata([frame_index, frame_index])
    return visual_images + visual_labels + visual_cursors

visual_preview_indices = sorted(set(visual_full) | set(np.linspace(0, len(visual_gray) - 1, min(16, len(visual_gray)), dtype=int)))
visual_animation = FuncAnimation(visual_fig, update_method_preview, frames=visual_preview_indices,
                                 interval=300, repeat=True, blit=False)
visual_html = visual_animation.to_jshtml()
plt.close(visual_fig)
display(HTML(visual_html))
display(Markdown('**Reading the visual:** Colored dots mark frames whose image descriptors are used. '
                 'The black line marks the preview frame. Dim images have no frame descriptor computed by that method. '
                 'The adaptive choice comes from the saved model for a fold that held out this clip group. '
                 'All-frame pooling and transition descriptors are not drawn. Both sampled methods still retrieve eight images; '
                 'the adaptive method saves descriptor work, not necessarily decoding. '
                 'The animation previews up to 24 positions, not every video frame; the timelines show the full selection. '
                 'This display decodes the whole clip for illustration and is not part of the speed benchmark. '
                 'Change VIS_LANGUAGE to Nepali or VIS_CLIP to another test clip to compare a different sample.'))

## What the data can tell us

ISL has 412 clips in eight word classes. Five-fold tests keep matching filename IDs together. These IDs are **not verified signer identities**.

Nepali has 15 alphabet clips in five classes. Ten train clips come from S2/S3; five test clips come from S1. One Nepali mistake changes accuracy by 20 percentage points. This is a tiny demonstration, not a reliable language benchmark.

Languages are trained and tested separately. Their labels and task difficulty differ, so we do not pool them or claim cross-language transfer.

The adaptive threshold is chosen using a separate grouped validation split inside training. Final models are then refitted on all training clips. Test labels do not choose the gate. Nepali validation has only five clips, so its threshold is especially uncertain. Classifier settings are fixed; there is no long tuning search.

In [ ]:
for language, report in REPORTS.items():
    display(Markdown('## ' + language + ' accuracy results'))
    display(speed_poc.table(report, 'quality'))

In [ ]:
for language, report in REPORTS.items():
    display(Markdown('### What these numbers mean: ' + language + '\n' + speed_poc.explanation(report, 'quality')))

## Speed and work

Timing runs from opening a video through prediction. Each method runs independently, twice per test clip, in shuffled order. No descriptor cache is shared between methods. Files may be warm in the operating-system cache.

All methods traverse the entire compressed clip. Sampled methods retrieve and resize only eight selected images using OpenCV grab/retrieve. This saves image conversion and feature work. It does **not** guarantee that the codec skips decoding. Wrong frame-count metadata triggers a full-read fallback.

The adaptive method computes only four frame descriptors at first. On rescue, it reuses those descriptors. Extra transitions and a second classifier call can make rescued clips slower.

In [ ]:
for language, report in REPORTS.items():
    display(Markdown('## ' + language + ' speed results'))
    measured = speed_poc.table(report, 'speed')
    display(measured[['mean_ms', 'median_ms', 'p95_ms', 'clips_per_second', 'speedup_vs_all']])
    display(measured[['decode_ms', 'feature_policy_ms', 'model_ms', 'mean_frames', 'mean_hog_calls', 'rescue_rate']])
    display(measured[['mean_decoded', 'mean_retrieved', 'gray_buffer_kib', 'model_kib', 'training_seconds']])
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
    names = list(report['results'])
    axes[0].bar(names, [report['results'][name]['mean_ms'] for name in names])
    axes[0].set(ylabel='Milliseconds per clip', title=language + ' total time')
    axes[1].bar(names, [report['results'][name]['mean_hog_calls'] for name in names])
    axes[1].set(ylabel='Descriptor calls per clip', title='Actual feature work')
    for axis in axes:
        axis.tick_params(axis='x', rotation=15)
    fig.tight_layout()
    plt.show()

In [ ]:
for language, report in REPORTS.items():
    display(Markdown('### What these numbers mean: ' + language + '\n' + speed_poc.explanation(report, 'speed')))

In [ ]:
for language, report in REPORTS.items():
    display(Markdown('## ' + language + ' class results'))
    classes = report['classes']
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
    for axis, (method, result) in zip(axes, report['results'].items()):
        display(Markdown('### ' + method))
        display(pd.DataFrame({label: result['per_class'][label] for label in classes}).T.round(3))
        matrix = np.array(result['confusion_matrix'])
        axis.imshow(matrix, cmap='Blues')
        axis.set_title(method)
        axis.set_xticks(range(len(classes)), classes, rotation=60)
        axis.set_yticks(range(len(classes)), classes)
        axis.set(xlabel='Predicted', ylabel='True')
        for row, column in np.ndindex(matrix.shape):
            axis.text(column, row, str(matrix[row, column]), ha='center', va='center', fontsize=8)
    fig.suptitle(language)
    fig.tight_layout()
    plt.show()

In [ ]:
for language, report in REPORTS.items():
    candidate = report['results']['Adaptive 4 to 8']
    weakest = min(report['classes'], key=lambda label: candidate['per_class'][label]['recall'])
    display(Markdown('### What the class results mean: ' + language + '\n'
                    'Each row is one sign. Support is the number of test examples. The adaptive method has its lowest recall for '
                    + weakest + '. Matrix rows show the real label; columns show the predicted label. '
                    'Diagonal counts are correct; other cells are mistakes. With one Nepali test clip per class, these class scores are not stable estimates.'))

In [ ]:
for language, report in REPORTS.items():
    display(Markdown('## ' + language + ' gate settings'))
    display(pd.DataFrame(report['folds'])[['fold', 'threshold', 'gate_validation_clips',
                                          'gate_validation_reject_rate', 'train_groups', 'test_groups']])

### What the gate means

The gate uses the gap between the two highest linear classifier scores. It is **not a probability**. We choose the threshold that asks for the least extra work while keeping validation macro F1 within 0.02 of uniform eight.

This is a validation rule, not a guarantee on new clips. Scores can change after refitting. Different folds can choose different thresholds. A threshold of zero means no rescue, even for wrong predictions. This is a weak point of the current prototype.

Train and test groups must not overlap. The gate is tested on new groups after its settings are chosen.

In [ ]:
for language, report in REPORTS.items():
    adaptive = report['results']['Adaptive 4 to 8']
    uniform = report['results']['Uniform 8']
    dense = report['results']['All frames']
    display(Markdown(
        '## Takeaway: ' + language + '\n'
        + f"Adaptive accuracy: {adaptive['accuracy']:.1%}. Uniform accuracy: {uniform['accuracy']:.1%}. "
        + f"Descriptor-frame reduction against uniform: {1 - adaptive['mean_frames'] / uniform['mean_frames']:.1%}. "
        + f"Measured speed ratio against uniform: {uniform['mean_ms'] / adaptive['mean_ms']:.2f}x. "
        + f"Against all frames: {dense['mean_ms'] / adaptive['mean_ms']:.2f}x. "
        + 'Large gains against all frames also come from sparse retrieval, which uniform eight shares. '
        + 'The candidate does not beat every baseline in accuracy. A tiny latency difference can be timing noise. '
        + 'This is a low-work prototype, not a proven new state of the art.'
    ))

## Limits and what was not measured

We measured accuracy, balanced accuracy, macro precision/recall/F1, weighted F1, kappa, MCC, per-class support, confusion matrices, mean/median/p95 latency, serial throughput, decode/feature/model time, frame and descriptor work, rescue rate, stored pixel bytes, model storage, and classifier training time.

Energy, FLOPs and peak RAM were not measured. Buffer size is stored grayscale pixels, not peak RAM. Model size is pickle storage, not peak RAM. Hinge scores are not probabilities, so probability-based log loss and calibration metrics are not reported. Training time excludes shared feature preparation. No statistical-significance claim is made.

The candidate has two classifiers and takes more model storage than uniform eight. Using fewer frames does not make every cost smaller.

The all-frame method averages frame features into eight bins. Sampled methods keep ordered selected features. This is a practical pipeline comparison, not an experiment that changes frame count alone.

Whole-image features may learn background shortcuts. We need more Nepali clips, verified ISL signer identities, and an untouched larger test set. The local Nepali subset tests alphabet signs, whereas ISL tests words.

Raw predictions, timing rows, group logs and trained fold models are saved under `results/speed_poc`. Only load trusted local pickle files. The original long notebook and its results were left untouched.

## Research and credit

| Reference | What we use from it |
| --- | --- |
| [Dalal and Triggs, CVPR 2005: Histograms of Oriented Gradients for Human Detection](https://ieeexplore.ieee.org/document/1467360) | Gradient-histogram features. Our cell-normalized descriptor is simplified, not exact original block-normalized HOG. |
| [Wang et al., ECCV 2016: Temporal Segment Networks](https://arxiv.org/abs/1608.00859) | Sparse temporal sampling as an established idea. We do not reproduce the TSN neural network. |
| [Teerapittayanon, McDanel and Kung, ICPR 2016; arXiv 2017: BranchyNet](https://arxiv.org/abs/1709.01686) | Asking for more computation on difficult inputs. Our two-classifier cascade is not BranchyNet. |
| [Sunuwar, Borah and Kharga, Data in Brief 2024: NSL23 dataset for alphabets of Nepali sign language](https://doi.org/10.1016/j.dib.2024.109986) | Nepali dataset attribution. |

Nepali source card: [Titung/nepali-sign-language-video](https://huggingface.co/datasets/Titung/nepali-sign-language-video). The card states CC BY 4.0. Keep the attribution and file mapping in the local `External_Data_NS23` README and manifest.

The candidate contribution is a small four-to-eight-frame sign cascade with feature reuse and a validation-based speed/accuracy rule, evaluated on two local datasets. HOG, sparse sampling, and conditional inference are established. A focused literature review is still needed before claiming the exact combination is new. Local ISL provenance has not been independently verified.

The standalone helper is `speed_poc.py`. Minimal dependencies are listed in `requirements-speed.txt`. The selected Python 3.9 environment already has them installed. To recalculate all results, change `RERUN` to `True` in Cell 2 and run all. Leave it `False` for a quick presentation of the completed run.